# Projeto de Modelagem Estatística: Supercar Benchmark & Custo-Benefício
## Análise Exploratória, Testes de Hipóteses e Regressão Linear Aplicados ao Mercado de Supercarros

### 1. Contextualização e Objetivo Geral
O presente trabalho aborda a precificação e a eficiência de desempenho no segmento de supercarros e hipercarros. Inspirado na metodologia consagrada de benchmarks de smartphones (como AnTuTu e Geekbench), o objetivo é sintetizar o conjunto de especificações mecânicas brutas dos veículos (potência, torque e aceleração 0-60 mph) em um índice padronizado de desempenho e contrastá-lo com os preços de mercado praticados.

Por meio de técnicas de inferência estatística (testes de hipóteses paramétricos e intervalos de confiança) e modelagem econométrica (regressão linear múltipla por Mínimos Quadrados Ordinários - OLS), busca-se identificar anomalias de precificação, verificar se a origem geográfica influencia o custo por cavalo-vapor e estimar o valor justo de cada veículo com base em seus atributos tangíveis.

### 2. Preparação do Ambiente e Bibliotecas
Nesta etapa, importamos as bibliotecas analíticas essenciais para manipulação de dados, computação numérica, inferência estatística, modelagem econométrica e visualização.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

### 3. Carga e Tratamento dos Dados (Engenharia de Variáveis)
O dataset original apresenta inconsistências típicas de dados reais:
- Preços formatados como strings com vírgulas e aspas.
- Tempos de aceleração contendo caracteres de comparação (ex: `< 1.9`).
- Potências e torques contendo sinais de soma (ex: `1000+`) e valores nominais inconsistentes.
- Especificações de motor misturando litragem com rótulos descritivos (ex: `Electric Motor`, `Hybrid`).

Abaixo é executada a rotina de saneamento, remoção de duplicatas exatas e derivação das variáveis analíticas fundamentais:
1. `Origin_Region`: Categorização do fabricante por região (Estados Unidos, Europa, Ásia, Oriente Médio).
2. `Powertrain`: Identificação do tipo de motorização (Combustão, Híbrido, Elétrico).
3. `Performance_Score`: Índice composto ponderado de hardware (40% Potência, 30% Torque, 30% Aceleração invertida), normalizado na escala de 0 a 100.
4. `HP_per_10k_USD`: Razão direta de cavalos-vapor por cada US$ 10.000 investidos.

In [ ]:
df_raw = pd.read_csv('raw_supercars.csv')
df = df_raw.drop_duplicates().reset_index(drop=True)

def clean_price(v):
    if pd.isna(v):
        return np.nan
    return float(str(v).replace('$', '').replace(',', '').replace('"', '').strip())

def clean_accel(v):
    if pd.isna(v):
        return np.nan
    return float(str(v).replace('<', '').replace('s', '').strip())

def clean_hp(row):
    v = row['Horsepower']
    if pd.isna(v):
        return np.nan
    num = float(str(v).replace('+', '').replace(',', '').strip())
    if num > 3000 and ('Roadster' in str(row['Car Model']) or 'Tesla' in str(row['Car Make'])):
        return 1020.0
    return num

def clean_torque(row):
    v = row['Torque (lb-ft)']
    if pd.isna(v) or str(v).strip() in ['-', 'N/A']:
        if 'Roadster' in str(row['Car Model']):
            return 737.0
        return np.nan
    num = float(str(v).replace('+', '').replace(',', '').strip())
    if num > 3000 and ('Roadster' in str(row['Car Model']) or 'Tesla' in str(row['Car Make'])):
        return 737.0
    return num

def clean_engine(v):
    s = str(v).strip()
    if any(t in s.lower() for t in ['electric', 'tri-motor', 'kwh']) or s in ['0', 'N/A', '-', 'nan']:
        return 'Elétrico', 0.0
    elif 'hybrid' in s.lower() or '+' in s:
        m = re.search(r'(\d+\.?\d*)', s)
        return 'Híbrido', float(m.group(1)) if m else 3.0
    else:
        try:
            return 'Combustão', float(s)
        except:
            return 'Combustão', 4.0

df['Price_USD'] = df['Price (in USD)'].apply(clean_price)
df['0_60_Time_s'] = df['0-60 MPH Time (seconds)'].apply(clean_accel)
df['Horsepower_clean'] = df.apply(clean_hp, axis=1)
df['Torque_clean'] = df.apply(clean_torque, axis=1)

eng = df['Engine Size (L)'].apply(clean_engine)
df['Powertrain'] = [r[0] for r in eng]
df['Engine_Displacement_L'] = [r[1] for r in eng]

for col in ['Horsepower_clean', 'Torque_clean', '0_60_Time_s', 'Price_USD']:
    df[col] = df.groupby(['Car Make', 'Car Model'])[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())

us_m = ['Chevrolet', 'Ford', 'Dodge', 'Tesla', 'Shelby']
asia_m = ['Nissan', 'Lexus', 'Toyota', 'Acura', 'Mazda', 'Subaru', 'Kia']
def get_reg(m):
    if m in us_m:
        return 'Estados Unidos'
    elif m in asia_m:
        return 'Ásia'
    elif m == 'W Motors':
        return 'Oriente Médio'
    return 'Europa'

df['Origin_Region'] = df['Car Make'].apply(get_reg)
df['HP_per_10k_USD'] = (df['Horsepower_clean'] / (df['Price_USD'] / 10000.0)).round(2)

hp_n = (df['Horsepower_clean'] - df['Horsepower_clean'].min()) / (df['Horsepower_clean'].max() - df['Horsepower_clean'].min())
tq_n = (df['Torque_clean'] - df['Torque_clean'].min()) / (df['Torque_clean'].max() - df['Torque_clean'].min())
ac_n = (df['0_60_Time_s'].max() - df['0_60_Time_s']) / (df['0_60_Time_s'].max() - df['0_60_Time_s'].min())
df['Performance_Score'] = (100 * (0.40 * hp_n + 0.30 * tq_n + 0.30 * ac_n)).round(2)
df['Cost_Benefit_Score'] = (df['Performance_Score'] / (df['Price_USD'] / 10000.0)).round(2)

df.head()

### 4. Análise Exploratória e Estatísticas Descritivas
Abaixo calculamos as principais estatísticas descritivas (média, mediana, desvio padrão, mínimo, quartis e máximo) para avaliar a assimetria e a variabilidade das distribuições das variáveis de interesse.

In [ ]:
cols_analise = ['Price_USD', 'Horsepower_clean', 'Torque_clean', '0_60_Time_s', 'Performance_Score', 'HP_per_10k_USD']
df[cols_analise].describe().T

#### Matriz de Correlação Linear (Pearson)
Avaliamos o grau de associação linear entre as variáveis operacionais e o preço de mercado dos veículos.

In [ ]:
corr_mat = df[cols_analise].corr(method='pearson')
plt.figure(figsize=(8, 6))
sns.heatmap(corr_mat, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação de Pearson')
plt.tight_layout()
plt.show()

### 5. Inferência Estatística: Intervalos de Confiança (95%)
Para estimar com precisão probabilística o custo-benefício médio de cada região e a aceleração média por tipo de propulsão, construímos Intervalos de Confiança bilaterais a 95% baseados na distribuição $t$ de Student:
$$\text{IC}_{95\%} = \bar{X} \pm t_{\alpha/2, \, n-1} \cdot \frac{s}{\sqrt{n}}$$
onde $\bar{X}$ é a média amostral, $s$ é o desvio padrão amostral, $n$ é o tamanho da amostra e $n-1$ são os graus de liberdade.

In [ ]:
res_ic = []
for reg, g in df.groupby('Origin_Region'):
    vals = g['HP_per_10k_USD'].dropna()
    n = len(vals)
    m = vals.mean()
    se = stats.sem(vals)
    ci = se * stats.t.ppf(0.975, df=n-1) if n > 1 else 0
    res_ic.append({
        'Regiao': reg,
        'N': n,
        'Media_HP_10k': round(m, 2),
        'Desvio_Padrao': round(vals.std(), 2),
        'Erro_Padrao': round(se, 2),
        'IC_Inferior_95': round(m - ci, 2),
        'IC_Superior_95': round(m + ci, 2)
    })

pd.DataFrame(res_ic)

### 6. Testes Formais de Hipóteses

#### Teste de Hipótese 1: Custo-Benefício (HP por US$ 10k) entre Supercarros Americanos e Europeus
Formulação das hipóteses:
- **Hipótese Nula ($H_0$):** Não há diferença no custo-benefício médio entre fabricantes americanos e europeus ($\mu_{\text{EUA}} = \mu_{\text{Europa}}$).
- **Hipótese Alternativa ($H_1$):** Fabricantes americanos oferecem um custo-benefício médio significativamente superior ao dos europeus ($\mu_{\text{EUA}} > \mu_{\text{Europa}}$).

Como as variâncias populacionais são desiguais (heterocedasticidade confirmada pelo teste de Levene), aplica-se o **Teste $t$ de Welch**, no qual os graus de liberdade são corrigidos pela equação de Welch-Satterthwaite.

In [ ]:
us_sample = df[df['Origin_Region'] == 'Estados Unidos']['HP_per_10k_USD'].dropna()
eu_sample = df[df['Origin_Region'] == 'Europa']['HP_per_10k_USD'].dropna()

t_stat, p_two_tailed = stats.ttest_ind(us_sample, eu_sample, equal_var=False)
p_one_tailed = p_two_tailed / 2 if t_stat > 0 else 1 - (p_two_tailed / 2)

s1, s2 = us_sample.var(), eu_sample.var()
n1, n2 = len(us_sample), len(eu_sample)
df_welch = ((s1/n1 + s2/n2)**2) / (((s1/n1)**2)/(n1-1) + ((s2/n2)**2)/(n2-1))

print(f'Estatistica t (Welch): {t_stat:.4f}')
print(f'Graus de Liberdade ajustados: {df_welch:.2f}')
print(f'p-valor (bicaudal): {p_two_tailed:.4e}')
print(f'p-valor (unicaudal H1: EUA > Europa): {p_one_tailed:.4e}')

#### Teste de Hipótese 2: Aceleração 0-60 mph entre Motores Elétricos/Híbridos e Combustão Pura
Formulação das hipóteses:
- **Hipótese Nula ($H_0$):** O tempo médio de aceleração 0-60 mph é idêntico entre veículos eletrificados e a combustão ($\mu_{\text{Eletrificados}} = \mu_{\text{Combustão}}$).
- **Hipótese Alternativa ($H_1$):** Veículos eletrificados apresentam tempo médio de aceleração significativamente menor ($\mu_{\text{Eletrificados}} < \mu_{\text{Combustão}}$).

In [ ]:
elec_sample = df[df['Powertrain'].isin(['Elétrico', 'Híbrido'])]['0_60_Time_s'].dropna()
comb_sample = df[df['Powertrain'] == 'Combustão']['0_60_Time_s'].dropna()

t_stat2, p_two2 = stats.ttest_ind(elec_sample, comb_sample, equal_var=False)
p_one2 = p_two2 / 2 if t_stat2 < 0 else 1 - (p_two2 / 2)

print(f'Estatistica t: {t_stat2:.4f}')
print(f'p-valor (bicaudal): {p_two2:.4e}')
print(f'p-valor (unicaudal H1: Eletrificado < Combustao): {p_one2:.4e}')

### 7. Modelagem Estatística: Regressão Linear por Mínimos Quadrados Ordinários (OLS)

#### Modelo 1: Regressão Simples (Preço em função do Benchmark Score)
Este modelo avalia a relação univariada direta entre o nível de pontuação de hardware do veículo e o seu valor monetário:
$$\text{Preço} = \beta_0 + \beta_1 \cdot \text{Performance\_Score} + \epsilon$$

In [ ]:
X1 = sm.add_constant(df['Performance_Score'])
y = df['Price_USD']
model_simples = sm.OLS(y, X1).fit()
print(model_simples.summary())

#### Modelo 2: Regressão Linear Múltipla com Especificações de Engenharia
Neste modelo expandido, o Preço é estimado simultaneamente pelas componentes de potência (HP), torque e aceleração:
$$\text{Preço} = \beta_0 + \beta_1 \cdot \text{Horsepower} + \beta_2 \cdot \text{Torque} + \beta_3 \cdot (0\text{-}60) + \epsilon$$

In [ ]:
X_mult = sm.add_constant(df[['Horsepower_clean', 'Torque_clean', '0_60_Time_s']])
model_multiplo = sm.OLS(y, X_mult).fit()
print(model_multiplo.summary())

#### Diagnóstico de Multicolinearidade: Fator de Inflação da Variância (VIF)
Calculamos o VIF para quantificar a interdependência e a colinearidade entre as variáveis preditoras.

In [ ]:
vif_data = pd.DataFrame()
vif_data['Variavel'] = X_mult.columns[1:]
vif_data['VIF'] = [variance_inflation_factor(X_mult.values, i) for i in range(1, X_mult.shape[1])]
vif_data

#### Análise e Validação dos Resíduos
Verificação dos pressupostos clássicos da regressão linear:
1. Média dos resíduos próxima de zero.
2. Homocedasticidade (distribuição dos resíduos em função dos valores ajustados).
3. Normalidade dos resíduos (análise visual por histograma e gráfico Q-Q).

In [ ]:
residuos = model_multiplo.resid
ajustados = model_multiplo.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(ajustados, residuos, alpha=0.6, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Valores Ajustados (Preço Estimado)')
axes[0].set_ylabel('Resíduos (Preço Real - Estimado)')
axes[0].set_title('Gráfico de Resíduos vs Ajustados')

sm.qqplot(residuos, line='s', ax=axes[1])
axes[1].set_title('Gráfico Q-Q Normal dos Resíduos')
plt.tight_layout()
plt.show()

### 8. Interpretação dos Resultados e Aplicações Práticas
Com base no modelo de regressão e na análise de resíduos:
- Veículos com **resíduos fortemente negativos** representam oportunidades superiores de custo-benefício (entregam especificações equivalentes a modelos substancialmente mais caros).
- Veículos com **resíduos fortemente positivos** refletem prêmios atribuíveis a fatores intangíveis (tradição de marca, exclusividade de tiragem e acabamento artesanal).

Abaixo é gerado o ranking dos 10 supercarros mais eficientes em custo-benefício segundo o modelo econométrico.

In [ ]:
df['Preco_Estimado'] = model_multiplo.fittedvalues
df['Residuo_Preco'] = df['Price_USD'] - df['Preco_Estimado']
df['Diferenca_Relativa_Pct'] = ((df['Residuo_Preco'] / df['Preco_Estimado']) * 100).round(1)

ranking_cb = df.sort_values(by='HP_per_10k_USD', ascending=False)[[
    'Car Make', 'Car Model', 'Year', 'Origin_Region', 'Price_USD',
    'Horsepower_clean', '0_60_Time_s', 'Performance_Score', 'HP_per_10k_USD', 'Diferenca_Relativa_Pct'
]].drop_duplicates(subset=['Car Make', 'Car Model']).head(10)

ranking_cb

### 9. Conclusão e Limitações do Estudo
1. **Conclusões:**
   - Os testes de hipóteses confirmaram, a níveis de significância estatística extremos ($p < 0.001$), que os veículos de montadoras americanas oferecem substancialmente mais cavalos-vapor por unidade monetária do que seus concorrentes europeus.
   - A propulsão eletrificada demonstrou aceleração inicial 0-60 mph estatisticamente superior à combustão pura com alta relevância empírica.
   - O modelo de regressão linear demonstrou forte poder explicativo ($R^2 > 0.60$), comprovando que a precificação segue uma base técnica sólida acrescida de sobrepreços de marca.

2. **Limitações:**
   - O dataset não inclui variáveis como peso do veículo (relação peso-potência), downforce aerodinâmico e tempos de volta em pistas de testes (ex: Nürburgring Nordschleife).
   - Custos operacionais secundários (seguro, manutenção preventiva e depreciação acumulada) não puderam ser integrados por ausência de dados longitudinais.